# Approche supervisée — SVM vs XGBoost

**Fichier :** `users_labeled_manual.csv`

**Cible :** `label` — 0 normal / 1 atypique (~17 %)

**Pipeline :** 8 features (hors règles) -> `StandardScaler` -> **ACP (5 comp.)** -> SVM / XGBoost

Labellisation : `docs/LABELISATION.md`

---

### Limite méthodologique

Les labels découlent de 4 règles Excel sur les mêmes variables. L'évaluation **exclut 8 features** liées aux règles.

**Annexe (section 8) :** 16 features + ACP 7 comp. — F1 ~ 0,970 (circularité).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = "users_labeled_manual.csv"
RANDOM_STATE = 42

In [ ]:
# Chargement des données étiquetées
df = pd.read_csv(DATA_PATH)

print("Shape :", df.shape)
print("\nColonnes :", df.columns.tolist())
print("\nDistribution des labels :")
print(df["label"].value_counts())
print("\nProportions :")
print(df["label"].value_counts(normalize=True).round(4) * 100)

df.head()

## 2. Préparation des features

**Features exclues** — présentes dans les règles Excel ou dérivées directes :
`retweet_ratio`, `avg_urls`, `avg_mentions`, `active_days`, `nb_tweets`, `followers_friends_ratio`, `nb_retweets`, `tweet_frequency`

**Features retenues (8)** — hors règles de labellisation :
`followers_count`, `friends_count`, `avg_tweet_length`, `avg_hashtags`, `avg_favorites`, `avg_retweet_count`, `verified`, `default_profile_image`

`label` = cible · `anomaly_score` = score Excel — **tous deux exclus des features**.

In [ ]:
# Features impliquées dans les règles Excel (docs/LABELISATION.md)
FEATURES_RULES = [
    "retweet_ratio", "avg_urls", "avg_mentions",
    "active_days", "nb_tweets", "followers_friends_ratio",
    "nb_retweets", "tweet_frequency",
]

features = [
    "followers_count", "friends_count",
    "avg_tweet_length", "avg_hashtags",
    "avg_favorites", "avg_retweet_count",
    "verified", "default_profile_image",
]

df["verified"] = df["verified"].astype(int)
df["default_profile_image"] = df["default_profile_image"].astype(int)

X = df[features].copy()
y = df["label"].copy()

print(f"Features retenues : {len(features)}")
print(features)
print(f"\nFeatures exclues (règles) : {len(FEATURES_RULES)}")
print(FEATURES_RULES)
print(f"\nValeurs manquantes : {X.isnull().sum().sum()}")
print(f"Shape X : {X.shape}")

In [ ]:
# Séparation train / test (80/20) avec stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape[0]:,} lignes")
print(f"Test  : {X_test.shape[0]:,} lignes")
print(f"\nDistribution train :\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nDistribution test :\n{y_test.value_counts(normalize=True).round(4)}")

# Normalisation — fit sur train uniquement
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nDonnées normalisées — train {X_train_scaled.shape}, test {X_test_scaled.shape}")

## 3. Réduction dimensionnelle (ACP)

Avec **8 features** en entrée, la variance cumulée atteint **100 % à 5 composantes** (PC6-PC7 n'ajoutent rien).

**Règle supervisée :** saturation à **99,9 %** de variance sur le train → **5 composantes**.

Le non supervisé utilise **7 composantes** (16 features, seuil 75 % ~ 79 %). Les deux choix sont indépendants.

| Contexte | Features | Composantes ACP |
|----------|----------|------------------|
| Non supervisé | 16 | 7 (~79 %) |
| **Supervisé (retenu)** | **8** | **5 (100 %)** |

In [ ]:
# ACP supervisée : saturation variance sur train (8 features -> 5 comp.)
N_COMPONENTS_16 = 7  # référence non supervisé / annexe (16 features)

pca_full = PCA()
pca_full.fit(X_train_scaled)
variance_cumulee = np.cumsum(pca_full.explained_variance_ratio_) * 100

N_COMPONENTS = int(np.searchsorted(variance_cumulee, 99.9) + 1)

print(f"Saturation 99,9 % variance (train, {X_train_scaled.shape[1]} features) : {N_COMPONENTS} composantes")
print(f"Variance cumulée retenue : {variance_cumulee[N_COMPONENTS-1]:.2f} %")
print(f"Référence non supervisé (16 feat.) : {N_COMPONENTS_16} composantes (~79 %)")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(variance_cumulee) + 1), variance_cumulee, "bo-")
ax.axhline(y=70, color="r", linestyle="--", label="70 %")
ax.axhline(y=80, color="g", linestyle="--", label="80 %")
ax.axvline(x=N_COMPONENTS, color="orange", linestyle=":", linewidth=2,
           label=f"Retenu = {N_COMPONENTS} ({variance_cumulee[N_COMPONENTS-1]:.1f} %)")
ax.set_xlabel("Nombre de composantes")
ax.set_ylabel("Variance expliquée cumulée (%)")
ax.set_title("ACP — Variance cumulée (train, 8 features)")
ax.legend()
ax.grid(True)
plt.show()

pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"\nShape après ACP : train {X_train_pca.shape}, test {X_test_pca.shape}")
print(f"Variance expliquée ({N_COMPONENTS} comp.) : {sum(pca.explained_variance_ratio_)*100:.2f} %")


## 4. Modèle 1 — SVM (Support Vector Machine)

On utilise `LinearSVC`, adapté aux grands jeux de données (> 500 000 lignes).  
`class_weight='balanced'` compense le déséquilibre des classes (~17 % de profils atypiques).

In [ ]:
print("Entraînement SVM en cours...")

t0 = time.time()
svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=3000,
    random_state=RANDOM_STATE
)
svm_model.fit(X_train_pca, y_train)
svm_train_time = time.time() - t0

svm_pred = svm_model.predict(X_test_pca)
svm_proba = svm_model.decision_function(X_test_pca)

print(f"Temps d'entraînement : {svm_train_time:.1f} s")
print("\nRapport de classification — SVM :")
print(classification_report(y_test, svm_pred, target_names=["Normal", "Atypique"]))

## 5. Modèle 2 — XGBoost

XGBoost gère nativement les données tabulaires volumineuses.  
`scale_pos_weight` est calibré sur le ratio classe majoritaire / classe minoritaire pour gérer le déséquilibre.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Entraînement XGBoost en cours...")

t0 = time.time()
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train_pca, y_train)
xgb_train_time = time.time() - t0

xgb_pred = xgb_model.predict(X_test_pca)
xgb_proba = xgb_model.predict_proba(X_test_pca)[:, 1]

print(f"Temps d'entraînement : {xgb_train_time:.1f} s")
print("\nRapport de classification — XGBoost :")
print(classification_report(y_test, xgb_pred, target_names=["Normal", "Atypique"]))

## 6. Comparaison des deux modèles (ACP 5 comp. + 8 features hors règles)

Métriques retenues :
- **F1-score (classe Atypique)**
- **Recall (Atypique)**
- **ROC-AUC**

> Entrée des modèles : espace ACP (5 composantes, 100 % variance avec 8 features).

In [ ]:
def compute_metrics(y_true, y_pred, y_score, train_time):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (Atypique)": precision_score(y_true, y_pred, pos_label=1),
        "Recall (Atypique)": recall_score(y_true, y_pred, pos_label=1),
        "F1 (Atypique)": f1_score(y_true, y_pred, pos_label=1),
        "ROC-AUC": roc_auc_score(y_true, y_score),
        "Temps entraînement (s)": round(train_time, 1)
    }

results = pd.DataFrame({
    "SVM": compute_metrics(y_test, svm_pred, svm_proba, svm_train_time),
    "XGBoost": compute_metrics(y_test, xgb_pred, xgb_proba, xgb_train_time)
}).T

print("=" * 60)
print(" TABLEAU COMPARATIF — SVM vs XGBoost")
print("=" * 60)
display(results.round(4))

In [ ]:
# Visualisation comparative des métriques clés
metrics_plot = ["F1 (Atypique)", "Recall (Atypique)", "Precision (Atypique)", "ROC-AUC"]
x = np.arange(len(metrics_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, results.loc["SVM", metrics_plot], width, label="SVM", color="#2196F3", edgecolor="black")
bars2 = ax.bar(x + width/2, results.loc["XGBoost", metrics_plot], width, label="XGBoost", color="#4CAF50", edgecolor="black")

ax.set_ylabel("Score")
ax.set_title("SVM vs XGBoost — 8 features hors règles de labellisation")
ax.set_xticks(x)
ax.set_xticklabels(metrics_plot)
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.4)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusion côte à côte
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, title in zip(
    axes,
    [svm_pred, xgb_pred],
    ["SVM — Matrice de confusion", "XGBoost — Matrice de confusion"]
):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", ax=ax,
                xticklabels=["Normal", "Atypique"],
                yticklabels=["Normal", "Atypique"])
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Réalité")
    ax.set_title(title, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Courbes ROC
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(y_test, svm_proba, name="SVM", ax=ax)
RocCurveDisplay.from_predictions(y_test, xgb_proba, name="XGBoost", ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Aléatoire")
ax.set_title("Courbes ROC — 8 features hors règles")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Importance des composantes principales — XGBoost
pc_labels = [f"PC{i+1}" for i in range(N_COMPONENTS)]
importance = pd.Series(xgb_model.feature_importances_, index=pc_labels).sort_values(ascending=True)
var_ratio = pd.Series(pca.explained_variance_ratio_, index=pc_labels) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

importance.plot(kind="barh", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Importance XGBoost — composantes ACP")
axes[0].set_xlabel("Importance")

var_ratio.sort_values().plot(kind="barh", ax=axes[1], color="coral", edgecolor="black")
axes[1].set_title("Variance expliquée par composante")
axes[1].set_xlabel("Variance (%)")

plt.tight_layout()
plt.show()

print("Top composantes (importance XGBoost) :")
print(importance.sort_values(ascending=False).head())

## 7. Sélection du modèle retenu

Critère : **meilleur F1-score sur la classe Atypique** (ACP 5 comp. + 8 features hors règles).

XGBoost reste meilleur que le SVM linéaire.

In [ ]:
best_model_name = results["F1 (Atypique)"].idxmax()
best_f1 = results.loc[best_model_name, "F1 (Atypique)"]
other_model = "XGBoost" if best_model_name == "SVM" else "SVM"

print("=" * 60)
print(" MODÈLE RETENU (8 features hors règles)")
print("=" * 60)
print(f"\nModèle retenu : {best_model_name}")
print(f"\nF1 (Atypique) : {best_f1:.4f}")
print(f"Recall (Atypique) : {results.loc[best_model_name, 'Recall (Atypique)']:.4f}")
print(f"ROC-AUC : {results.loc[best_model_name, 'ROC-AUC']:.4f}")
print(f"Temps d'entraînement : {results.loc[best_model_name, 'Temps entraînement (s)']} s")

print(f"\nComparaison avec {other_model} :")
print(f"  Δ F1       : {best_f1 - results.loc[other_model, 'F1 (Atypique)']:+.4f}")
print(f"  Δ Recall   : {results.loc[best_model_name, 'Recall (Atypique)'] - results.loc[other_model, 'Recall (Atypique)']:+.4f}")
print(f"  Δ ROC-AUC  : {results.loc[best_model_name, 'ROC-AUC'] - results.loc[other_model, 'ROC-AUC']:+.4f}")

if best_model_name == "XGBoost":
    print("\nJustification : XGBoost obtient le meilleur F1 hors règles de labellisation.")
    print("Le score modéré confirme que le label encode surtout les 4 critères Excel,")
    print("pas entièrement les 8 features conservées.")
else:
    print("\nJustification : Le SVM linéaire offre le meilleur F1 sur ce sous-ensemble.")

## 8. Annexe — effet de la circularité (16 features + ACP 7 comp.)

Avec les **16 features complètes** et **7 composantes** (règle non supervisée), XGBoost atteint F1 ~ **0,97**.

Ce n'est pas une preuve de généralisation, mais la confirmation que le label encode les règles Excel.

In [ ]:
FEATURES_ALL = [
    "followers_count", "friends_count", "followers_friends_ratio",
    "nb_tweets", "nb_retweets", "retweet_ratio",
    "avg_tweet_length", "avg_hashtags", "avg_urls", "avg_mentions",
    "avg_favorites", "avg_retweet_count",
    "active_days", "tweet_frequency", "verified", "default_profile_image",
]

X_all = df[FEATURES_ALL].copy()
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
scaler_a = StandardScaler()
X_train_a_s = scaler_a.fit_transform(X_train_a)
X_test_a_s = scaler_a.transform(X_test_a)
pca_a = PCA(n_components=N_COMPONENTS_16, random_state=RANDOM_STATE)
X_train_a_pca = pca_a.fit_transform(X_train_a_s)
X_test_a_pca = pca_a.transform(X_test_a_s)
scale_a = (y_train_a == 0).sum() / (y_train_a == 1).sum()

xgb_all = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_a, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
)
xgb_all.fit(X_train_a_pca, y_train_a)
pred_all = xgb_all.predict(X_test_a_pca)
f1_all = f1_score(y_test_a, pred_all)
f1_reduced = results.loc["XGBoost", "F1 (Atypique)"]

print("=" * 60)
print(" COMPARAISON — CIRCULARITÉ vs ÉVALUATION HONNÊTE")
print("=" * 60)
print(f"\nVariance ACP (16 feat., {N_COMPONENTS_16} comp.) : {sum(pca_a.explained_variance_ratio_)*100:.1f} %")
print(f"XGBoost — 16 features + ACP 7 comp. (circularité) : F1 = {f1_all:.4f}")
print(f"XGBoost —  8 features + ACP 5 comp. (hors règles)  : F1 = {f1_reduced:.4f}")
print("\n=> Supervisé : 5 comp. (8 feat.) · Annexe : 7 comp. (16 feat.)")
